# 01 — 古典経路計算

モデル：正式solver.enumerate_exactと独立validatorを直接import。入力：デポ＋固定3顧客、人工OD秒、時間窓・容量・サービス・帰還期限。出力：訪問順、道路旅行時間、可行性。

**実行方法**：`notebooks/README.md`の環境を用意し、上からRun All。作業ディレクトリはrepo rootまたはnotebooks。第三者の住所・大容量入力・SUMOは不要です。途中の条件を編集したら、そのセル以降を再実行してください。正式実験条件・freezeは変更しません。

**前提・範囲**：Python 3.12系で確認する小型教材／保存結果表示です。必要な配送を省略した電力量減を省エネ成果としません。量子実機・量子優位性・全需要充足・全システムエネルギーの実証ではありません。Colab実行は未検証です。


In [ ]:
from pathlib import Path
import sys, json, math
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/adapters.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリrootまたはnotebooksを作業ディレクトリにしてください')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'notebooks'))
import adapters as api
S = '01_research_design/scenario_evaluation/'
M = '01_research_design/methodology/'
Q = '05_src/traffic_simulation/'
L = '05_src/literature_analysis/'


In [ ]:
solver = api.load_module('src/kmd_quantitative/routing/solver.py','nb_solver')
validator = api.load_module('src/kmd_quantitative/routing/independent.py','nb_validator')
a = api.instance()
display(pd.DataFrame(a['travel'],index=a['nodes'],columns=a['nodes']))
display(pd.DataFrame({k:a[k] for k in ['nodes','demand','ready','due','service']}))
print('容量216、帰還期限32400秒。単一車両・固定割当。')

In [ ]:
route, stats = solver.enumerate_exact(a)
assert route is not None
checked = validator.validate(a,route)
assert route == [0,1,2,3,0] and Decimal(checked['totals']['road_travel_seconds']) == 40
assert validator.exact_dfs(a)[0] == route
print('訪問順:',route,'全順列:',stats)
display(pd.DataFrame([checked['totals']]))
display(pd.DataFrame(checked['stops']))

## 条件変更
顧客Bの時間窓上限を120秒へ変更します。元の40秒経路はBへのサービス開始122秒で制約違反となり、別順序が必要です。列挙関数単体は容量を検査しないため、必ず独立validatorと組み合わせます。

In [ ]:
changed = {k:v.copy() for k,v in a.items()}
changed['due'][2] = 120  # 編集可能、秒
changed_route, changed_stats = solver.enumerate_exact(changed)
if changed_route is None:
    print('可行経路なし：0Whや省エネ成果へ置換しません')
else:
    result = validator.validate(changed,changed_route)
    display(pd.DataFrame([{'route':changed_route,**result['totals']}]))
# 入力不能の負例
bad = {k:v.copy() for k,v in a.items()}; bad['due'][1] = 0
assert solver.enumerate_exact(bad)[0] is None
over = {k:v.copy() for k,v in a.items()}; over['demand'][1] = 217
try: validator.validate(over,route)
except AssertionError: print('容量違反を独立検査で検出')
else: raise AssertionError('容量違反未検出')

## 解釈と限界
期待40秒は4辺×最小10秒の下界を唯一の巡回が達成することから独立に確認できます。目的は道路旅行時間であり電力量ではありません。実研究の3/4/5顧客側の全列挙と、25/99/125顧客側のヒューリスティック＋下界評価は別です。ここでは実住所・道路ネットワーク・車両割当最適化を再現しません。